<a href="https://colab.research.google.com/github/26b21a4544kietgroup-spec/charan-D/blob/main/Backend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import numpy as np
import pandas as pd

In [2]:
x_supports = np.array([0.0, 10.0, 20.0])   # support positions (m)
x_loads    = np.array([4.0, 14.0, 15.0])   # load positions (m); 14.0 is the truck
P          = np.array([50.0, 120.0, 60.0]) # load magnitudes (kN)

In [3]:
def build_A(x_sup):
    return np.array([[1.0, 1.0, 1.0],      # vertical balance
                     list(x_sup),          # moment about x = 0
                     [1.0, -2.0, 1.0]])    # continuity condition

def build_B(x_ld, P_ld):
    return np.array([P_ld.sum(),           # total load
                     (P_ld * x_ld).sum(),  # total moment
                     0.0])

A = build_A(x_supports)
B = build_B(x_loads, P)

print("A =\n", A)
print("B =", B)

A =
 [[ 1.  1.  1.]
 [ 0. 10. 20.]
 [ 1. -2.  1.]]
B = [ 230. 2780.    0.]


In [4]:
detA = np.linalg.det(A)
print("det(A) =", round(detA, 2))
assert abs(detA) > 1e-9, "Singular matrix: no unique solution"

det(A) = 60.0


In [5]:
R = np.linalg.solve(A, B)          # same as inv(A) @ B, but more stable
R1, R2, R3 = R
print(f"R1 = {R1:.2f} kN\nR2 = {R2:.2f} kN\nR3 = {R3:.2f} kN")

R1 = 52.67 kN
R2 = 76.67 kN
R3 = 100.67 kN


In [6]:
print("Residual |AR - B| =", np.abs(A @ R - B).max())      # should be about 0
print("Sum of R =", round(R.sum(), 2), "| Total load =", P.sum())
print("Balanced:", np.isclose(R.sum(), P.sum()))

Residual |AR - B| = 1.4210854715202004e-14
Sum of R = 230.0 | Total load = 230.0
Balanced: True


In [7]:
A_inv = np.linalg.inv(A)
R_inv = A_inv @ B
print("A inverse =\n", np.round(A_inv, 4))
print("R = A^-1 B =", np.round(R_inv, 2))

A inverse =
 [[ 0.8333 -0.05    0.1667]
 [ 0.3333  0.     -0.3333]
 [-0.1667  0.05    0.1667]]
R = A^-1 B = [ 52.67  76.67 100.67]


In [8]:
def reactions(truck_x, other=((4.0, 50.0), (15.0, 60.0)), truck_P=120.0):
    xl = np.array([other[0][0], truck_x, other[1][0]])
    pl = np.array([other[0][1], truck_P, other[1][1]])
    return np.linalg.solve(build_A(x_supports), build_B(xl, pl))

In [9]:
positions = [0, 4, 8, 12, 14, 16, 20]
rows = [[x, *np.round(reactions(x), 2)] for x in positions]
df = pd.DataFrame(rows, columns=["Truck x (m)", "R1 (kN)", "R2 (kN)", "R3 (kN)"])
df["Sum (kN)"] = df[["R1 (kN)", "R2 (kN)", "R3 (kN)"]].sum(axis=1).round(2)
df

,Truck x (m),R1 (kN),R2 (kN),R3 (kN),Sum (kN)
0,0,136.67,76.67,16.67,230.01
1,4,112.67,76.67,40.67,230.01
2,8,88.67,76.67,64.67,230.01
3,12,64.67,76.67,88.67,230.01
4,14,52.67,76.67,100.67,230.01
5,16,40.67,76.67,112.67,230.01
6,20,16.67,76.67,136.67,230.01
